# LangChain Dasar: dari prompt ke RAG dan agent kecil

**Walkthrough hands-on · Pemula · sekitar 105 menit**

Kita akan belajar model, prompt, dan chain; melihat RunnableParallel dalam mini RAG; lalu membuat asisten kedai kopi yang memakai tool dan mengumpulkan pesanan melalui percakapan.

**Hasil akhir:** sebuah chain sederhana, mini RAG lokal, dan agent kedai kopi yang menghitung pesanan dari hasil tool.

```text
Chain:  input → prompt → model → parser → jawaban
Agent:  pesan → model memilih tool → tool Python berjalan
                          ↑                 ↓
                          └── hasil tool ←──┘
                                  ↓
                            jawaban akhir
```

> **Batas latihan:** Menu adalah data contoh di notebook. Tool hanya menghitung dan tidak mengubah data. Permintaan model dikirim ke penyedia API; gunakan pertanyaan latihan, bukan informasi pribadi atau rahasia.

## 1. Tujuan, prasyarat, dan istilah

Setelah latihan ini, Anda dapat:

1. Menjelaskan peran model, prompt template, chain, parser, tool, dan agent.
2. Menyusun chain dengan alur `prompt | model | parser`.
3. Membungkus fungsi Python menjadi tool dengan `@tool`.
4. Membaca riwayat pesan untuk melihat permintaan tool, hasil tool, dan jawaban agent, serta mempertahankan konteks percakapan multi-turn.
5. Menyusun mini RAG dengan `RunnableParallel` dan menambahkan `context` serta `answer` memakai `RunnablePassthrough.assign`.
6. Menguji batas sederhana dan menjelaskan mengapa notebook ini belum siap produksi.

**Prasyarat:** dasar Python (fungsi, dictionary, tipe data), bisa menjalankan notebook Jupyter, dan akses API OpenAI. Python 3.11 atau lebih baru disarankan.

| Istilah | Arti singkat |
|---|---|
| Model | Komponen yang menghasilkan atau memilih respons berdasarkan input. |
| Prompt template | Pola instruksi dengan tempat untuk nilai yang berubah. |
| Chain | Beberapa langkah tetap yang dijalankan berurutan. |
| Retriever | Komponen yang mencari dokumen relevan untuk pertanyaan. |
| RunnableParallel | Menjalankan beberapa cabang dengan input yang sama secara bersamaan. |
| `assign` | Menambahkan hasil baru ke dictionary tanpa membuang field sebelumnya. |
| Tool | Fungsi yang bisa dipanggil agent untuk mengambil data atau melakukan pekerjaan tertentu. |
| Agent | Alur yang meminta model memilih langkah, menjalankan tool yang diizinkan, lalu mengembalikan hasilnya ke model. |

**Rencana waktu:** pengantar 10 menit · setup 10 menit · model dan chain 15 menit · mini RAG dengan RunnableParallel 15 menit · tool dan agent 20 menit · pesanan multi-turn 15 menit · tantangan dan refleksi 15 menit · rangkuman 5 menit (sekitar 105 menit).

## 2. Siapkan Environment

Dari folder `modules/20-Langchain`, buat virtual environment dan pasang dependensi. Perintah ini hanya perlu dijalankan sekali.

**Windows PowerShell**

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install -r requirements-notebook.txt
python -m jupyter lab
```

**macOS atau Linux**

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements-notebook.txt
python -m jupyter lab
```

Buka notebook ini dan pastikan kernel Jupyter memakai virtual environment tersebut.

> **Catatan biaya:** Latihan ini membuat beberapa permintaan ke API model dan penggunaannya dapat dikenai biaya berdasarkan pemakaian. Buat API key untuk latihan, jangan tulis key ke notebook, dan periksa tarif penyedia sebelum kelas. LangSmith tracing tidak diperlukan.

## 3. Impor komponen dan siapkan model

LangChain menyediakan objek model dengan cara pemanggilan yang seragam. Di contoh ini kita menggunakan adapter OpenAI; konsep prompt, chain, dan tool tetap dapat digunakan dengan adapter provider lain.

Masukkan API key saat diminta. `getpass` tidak menampilkan karakter yang diketik.

In [2]:
!pip install langchain==1.4.3 langchain-openai==1.6.3

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 12.8 MB/s  0:00:00
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.7/813.7 kB 27.5 MB/s  0:00:00
Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl (175 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.9/562.9 kB 22.3 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jsonpointer-3.1.1-py3-none-

In [23]:
import getpass
import os

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (input hidden): ")

MODEL_NAME = "gpt-5.6-luna"
# ini dari langchain ChatOpenAI()
model = ChatOpenAI(model=MODEL_NAME, temperature=1,reasoning={"effort": "medium"}) #kucoba edit temperature 1 biar halusinasi


print(f"Model siap pak eko: {MODEL_NAME}")

# dibagian cmd ini bakal minta API key, masukin aja dari noted di https://app.notion.com/p/Week1-Phase-2-3e9bfaebb15f80118b24c19d8317c906

Model siap pak eko: gpt-5.6-luna


### Checkpoint 1 — Panggil model secara langsung

Sebelum menambahkan komponen lain, kirim satu permintaan singkat ke model. `invoke()` menjalankan satu input dan mengembalikan objek pesan dari model.

**Prediksi sebelum menjalankan:** apa bedanya objek respons ini dengan sebuah string Python biasa?

In [7]:
# model.invoke() untuk call api openAI nya
response = model.invoke("Jelaskan LangChain dalam satu kalimat sederhana dalam Bahasa Indonesia.")

print(type(response).__name__)
print(response.content)

AIMessage
LangChain adalah kerangka kerja untuk membangun aplikasi berbasis model bahasa besar (LLM) dengan menghubungkannya ke data, alat, dan alur kerja.


**Yang perlu diperhatikan:** teks jawaban bisa berubah setiap kali dijalankan. Objek hasilnya adalah pesan AI; bagian teks yang biasanya ingin ditampilkan ada di `response.content`.

### Checkpoint 2 — Buat prompt template

Prompt template membantu kita menjaga instruksi tetap konsisten sambil mengganti input. Variabel ditulis dengan tanda kurung kurawal, seperti `{konsep}`.

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Kamu adalah mentor Python untuk pemula. Jawab singkat dan gunakan contoh sederhana."),
        ("human", "Jelaskan {konsep} dengan bahasa yang mudah dimengerti."),
        # ("ai","bisa juga di check di langchain reference") https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate
    ]
)

prompt_value = prompt.invoke({"konsep": "function dalam Python"})
prompt_value

ChatPromptValue(messages=[SystemMessage(content='Kamu adalah mentor Python untuk pemula. Jawab singkat dan gunakan contoh sederhana.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Jelaskan function dalam Python dengan bahasa yang mudah dimengerti.', additional_kwargs={}, response_metadata={})])

`prompt.invoke(...)` mengisi `{konsep}` dan membentuk pesan-pesan untuk model. Pada tahap ini model belum dipanggil.

### Checkpoint 3 — Rangkai sebuah chain

Operator `|` menghubungkan langkah-langkah yang berjalan berurutan:

1. `prompt` mengubah dictionary input menjadi pesan.
2. `model` menghasilkan objek pesan AI.
3. `StrOutputParser()` mengambil teks jawaban saja.

Chain ini selalu menjalankan langkah-langkah tersebut dalam urutan yang sama. Belum ada pemilihan tool.

In [11]:
prompt

ChatPromptTemplate(input_variables=['konsep'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Kamu adalah mentor Python untuk pemula. Jawab singkat dan gunakan contoh sederhana.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['konsep'], input_types={}, partial_variables={}, template='Jelaskan {konsep} dengan bahasa yang mudah dimengerti.'), additional_kwargs={})])

In [5]:
chain = prompt | model | StrOutputParser()

answer = chain.invoke({"konsep": "function dalam Python"})
print(answer)

Function dalam Python adalah sekumpulan kode yang dirancang untuk melakukan tugas tertentu. Dengan menggunakan function, kita bisa mengorganisir kode agar lebih rapi dan mudah digunakan kembali.

Contoh sederhana:

```python
def sapa(nama):
    print("Halo, " + nama + "!")

sapa("Budi")
```

Penjelasan:
- `def` adalah kata kunci untuk mendefinisikan function.
- `sapa` adalah nama function.
- `nama` adalah parameter yang bisa kita berikan saat memanggil function.
- Ketika kita memanggil `sapa("Budi")`, outputnya adalah "Halo, Budi!". 

Dengan function, kita bisa memanggil kode yang sama berkali-kali tanpa menulis ulang.


**Coba ubah:** ganti nilai `konsep` menjadi `dictionary` atau `API`. Perhatikan bahwa bentuk chain tetap sama, sementara isi prompt berubah.

### Ringkasan mini — chain

Chain cocok saat urutan kerja sudah diketahui. Ia tidak memutuskan sendiri apakah akan memanggil tool.

### Use case — mini RAG dengan `RunnableParallel`

Sekarang kita sambungkan bagian-bagian tadi menjadi retrieval-augmented generation (RAG). Untuk membuat alur mudah dilihat, kita gunakan tiga FAQ lokal dan retriever sederhana yang mencocokkan kata. **Ini bukan vector search** dan tidak memerlukan embeddings atau database.

```text
Pertanyaan
  ├─ question: teruskan pertanyaan apa adanya
  └─ documents: cari FAQ yang cocok
              ↓
{question, documents}
              ↓ assign context
{question, documents, context}
              ↓ assign answer = prompt → model → parser
{question, documents, context, answer}
```

`RunnableParallel` mengirim input yang sama ke dua cabang. `RunnablePassthrough.assign(...)` lalu menambahkan field ke dictionary sambil mempertahankan field yang sudah ada. Retriever dipanggil otomatis setiap kali chain berjalan; agent tidak memilih apakah retrieval perlu dilakukan.

In [12]:
import re
# RunnableLambda adalah object langchain
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

faq_documents = [
    Document(
        page_content="Harga latte adalah Rp32.000 per gelas.",
        metadata={"source": "menu"},
    ),
    Document(
        page_content=(
            "Minuman yang salah dibuat dapat diganti dalam 10 menit setelah diterima. "
            "Tunjukkan struk pesanan."
        ),
        metadata={"source": "kebijakan-penggantian"},
    ),
    Document(
        page_content="Kedai buka setiap hari pukul 08.00 sampai 20.00.",
        metadata={"source": "jam-operasional"},
    ),
]

STOP_WORDS = {"berapa", "lama", "yang", "saya", "bisa", "dapat", "dalam", "setelah"}


def retrieve_faq(question: str) -> list[Document]:
    """Return up to two FAQ documents with the most words in common with the question."""
    question_terms = set(re.findall(r"\w+", question.casefold())) - STOP_WORDS
    scored_documents = []

    for document in faq_documents:
        document_terms = set(re.findall(r"\w+", document.page_content.casefold())) - STOP_WORDS
        score = len(question_terms & document_terms)
        if score > 0:
            scored_documents.append((score, document))

    scored_documents.sort(key=lambda pair: pair[0], reverse=True)
    return [document for _, document in scored_documents[:2]]


retriever = RunnableLambda(retrieve_faq)

### Bangun dan jalankan RAG chain

Coba perhatikan tiga batas `|` di bawah:

1. `RunnableParallel` mempertahankan pertanyaan dan mengambil dokumen.
2. `assign(context=...)` mengubah dokumen menjadi teks konteks.
3. `assign(answer=prompt | model | parser)` membuat jawaban dari pertanyaan dan konteks.

Model hanya diminta menjawab dari konteks yang ditemukan. Dalam aplikasi nyata, tambahkan pemeriksaan kualitas retrieval dan penanganan jika tidak ada dokumen yang cocok.

In [14]:
def format_docs(documents: list[Document]) -> str:
    """Join retrieved document text into one context string."""
    return "\n\n".join(document.page_content for document in documents)

# ini document maa yag cocok sama pertanyaannya
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Jawab hanya berdasarkan konteks. Jika jawabannya tidak ada di konteks, "
            "katakan bahwa informasi belum tersedia.",
        ),
        ("human", "Konteks:\n{context}\n\nPertanyaan: {question}"),
    ]
)

rag_chain = (
    # step 1
    RunnableParallel(
        question=RunnablePassthrough(),
        documents_ya_bro=retriever,
    )
    # step 2
    | RunnablePassthrough.assign(
        context=lambda inputs: format_docs(inputs["documents_ya_bro"])
    )
    # step 3
    | RunnablePassthrough.assign(
        answer=rag_prompt | model | StrOutputParser()
    )
)

rag_question = "Berapa lama minuman yang salah dibuat dapat diganti?"
rag_result = rag_chain.invoke(rag_question)

print("Pertanyaan:", rag_result["question"])
print("Dokumen yang ditemukan:")
for document in rag_result["documents_ya_bro"]:
    print("-", document.page_content)
print("\nJawaban:", rag_result["answer"])

Pertanyaan: Berapa lama minuman yang salah dibuat dapat diganti?
Dokumen yang ditemukan:
- Minuman yang salah dibuat dapat diganti dalam 10 menit setelah diterima. Tunjukkan struk pesanan.

Jawaban: 10 menit setelah diterima.


**Checkpoint RAG:** hasil dictionary chain memiliki `question`, `documents`, `context`, dan `answer`. Untuk pertanyaan contoh, dokumen yang cocok menyebut penggantian dalam **10 menit** dan meminta pelanggan menunjukkan struk. Kalimat jawaban model bisa berbeda.

**Coba ubah:** tanyakan jam operasional kedai. Setelah itu tanyakan kebijakan parkir yang tidak ada dalam FAQ. Apakah retriever menemukan konteks? Apakah jawaban model tetap berpegang pada konteks?

> **Batas contoh:** retriever ini membandingkan kata secara sederhana. Untuk koleksi dokumen yang lebih besar, sistem nyata dapat memakai BM25, embeddings/vector search, filter metadata, atau gabungan strategi—dan tetap perlu evaluasi retrieval.

## 4. Definisikan tool Python

Kita gunakan menu kecil dengan harga tetap. Fungsi menghitung total dan menangani jumlah yang tidak valid atau nama minuman yang tidak ada.

Decorator `@tool` memberi LangChain informasi seperti nama, deskripsi, dan skema argumen. Deskripsi tool penting karena membantu model memutuskan kapan tool itu sesuai.

In [ ]:
from langchain.tools import tool

MENU = {
    "americano": 25_000,
    "latte": 32_000,
    "iced tea": 18_000,
    "brownies": 22_000,
}


def format_idr(amount: int) -> str:
    """Format angka sebagai rupiah dengan pemisah ribuan titik."""
    return f"Rp{amount:,}".replace(",", ".")

# @tool ini dipake untuk deterministik output, karena  AI nggk bisa understand htung2 an, karena di potong2 per token
@tool
def calculate_drink_total(drink: str, quantity: int) -> str:
    # ini dikasih tau buat ngapain
    """Hitung total harga satu jenis item pada menu kedai latihan.

    Args:
        drink: Nama minuman atau item menu, misalnya latte.
        quantity: Jumlah item yang dipesan, dari 1 sampai 20.
    """
    normalized_drink = drink.strip().casefold()

    if quantity < 1:
        return "Jumlah item harus minimal 1."
    if quantity > 20:
        return "Latihan ini membatasi pesanan maksimal 20 item sekaligus."
    if normalized_drink not in MENU:
        available_items = ", ".join(sorted(MENU))
        return f"Item tidak ditemukan. Menu latihan: {available_items}."

    unit_price = MENU[normalized_drink]
    total = unit_price * quantity
    return (
        f"{quantity} x {normalized_drink.title()} "
        f"({format_idr(unit_price)} per item) = {format_idr(total)}."
    )

### Checkpoint 4 — Jalankan tool tanpa agent

Sebelum model memilih tool, kita uji fungsi secara langsung. Perhitungan uang tetap dilakukan oleh kode biasa agar hasilnya deterministik.

In [16]:
print(calculate_drink_total.invoke({"drink": "latte", "quantity": 3}))

3 x Latte (Rp32.000 per item) = Rp96.000.


**Hasil yang diharapkan:** `3 x Latte (Rp32.000 per item) = Rp96.000.`

Coba juga hasil untuk input yang tidak valid. Karena fungsi lokal deterministik, hasil untuk input yang sama seharusnya sama setiap kali.

In [17]:
print(calculate_drink_total.invoke({"drink": "matcha", "quantity": 1}))
print(calculate_drink_total.invoke({"drink": "latte", "quantity": 0}))

Item tidak ditemukan. Menu latihan: americano, brownies, iced tea, latte.
Jumlah item harus minimal 1.


## 5. Buat agent dengan satu tool

Agent memakai model untuk memutuskan apakah tool perlu dipanggil. Jika model meminta tool, runtime agent menjalankan fungsi Python, memasukkan hasilnya ke riwayat pesan, lalu meminta model menyusun jawaban akhir.

Tool dalam latihan ini hanya membaca menu latihan dan menghitung harga. Agent tidak diberi akses ke web, file, database, atau tindakan yang mengubah data.

In [18]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """Kamu adalah asisten kedai latihan yang menjawab dalam Bahasa Indonesia.
Untuk pertanyaan harga atau total, gunakan calculate_drink_total.
Jangan menebak harga dari ingatan. Sampaikan hasil tool dengan ringkas.
Jika item atau jumlah ditolak oleh tool, jelaskan pesan penolakannya.
"""

agent = create_agent(
    model=model,
    tools=[calculate_drink_total],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent siap dengan satu tool:", calculate_drink_total.name)

Agent siap dengan satu tool: calculate_drink_total


### Checkpoint 5 — Minta agent menghitung pesanan

Minta total untuk satu item menu dan tentukan secara eksplisit bahwa agent perlu memakai tool. Angka pada jawaban akhir mungkin diformat berbeda, jadi periksa juga hasil tool pada riwayat pesan.

In [24]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Berapa total harga 3 latte? Gunakan tool menu untuk memastikan hitungannya.",
            }
        ]
    }
)

print(result["messages"][-1].content)

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Function tools with reasoning_effort are not supported for gpt-5.6-luna in /v1/chat/completions. To use function tools, use /v1/responses or set reasoning_effort to 'none'.", 'type': 'invalid_request_error', 'param': 'reasoning_effort', 'code': None}}

### Lihat langkah yang terjadi

Nilai `messages` memuat pesan pengguna, permintaan tool dari model, hasil tool, dan jawaban akhir. Tidak semua model menampilkan teks pada pesan yang meminta tool; perhatikan `tool_calls` dan objek `ToolMessage`.

In [12]:
for message in result["messages"]:
    print(f"\n{type(message).__name__}")

    tool_calls = getattr(message, "tool_calls", [])
    if tool_calls:
        print("Permintaan tool:", tool_calls)

    if type(message).__name__ == "ToolMessage":
        print("Hasil tool:", message.content)


HumanMessage

AIMessage
Permintaan tool: [{'name': 'calculate_drink_total', 'args': {'drink': 'latte', 'quantity': 3}, 'id': 'call_OUb1illSuqXV7Ux6b1CI8BqA', 'type': 'tool_call'}]

ToolMessage
Hasil tool: 3 x Latte (Rp32.000 per item) = Rp96.000.

AIMessage


**Diskusikan:** siapa yang memilih tool? Siapa yang menghitung total? Apa yang akan Anda periksa jika jawaban akhir menyebut jumlah yang berbeda dari hasil tool?

> **Intuisi agent:** model meminta suatu tindakan; kode aplikasi yang menjalankan tool yang memang diberikan kepadanya. Model tidak menjalankan fungsi Python secara langsung.

## 6. Lanjutkan percakapan: "Ada lagi?" lalu hitung total

Pesanan pelanggan sering datang beberapa bagian. Kita akan memberi agent konteks percakapan agar ia bisa mengumpulkan item, bertanya apakah masih ada tambahan, lalu menghitung semuanya setelah pelanggan menyatakan selesai.

Kita tambahkan satu tool yang menerima seluruh item sebagai dictionary, misalnya `{"latte": 2, "iced tea": 1}`. Checkpointer `InMemorySaver` mengingat pesan di dalam notebook ini; gunakan `thread_id` yang sama untuk kedua giliran percakapan.

```text
Pelanggan: Saya pesan 2 latte.
Agent: Ada lagi yang ingin dipesan?
Pelanggan: Tambahkan 1 iced tea dan 2 brownies. Itu saja, tolong hitung totalnya.
Agent: menjalankan tool → memberi rincian dan total pesanan
```

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver


@tool
def calculate_order_total(items: dict[str, int]) -> str:
    """Hitung total semua item menu setelah pelanggan menyelesaikan pesanannya.

    Args:
        items: Peta nama item menu ke jumlah, misalnya {"latte": 2, "iced tea": 1}.
    """
    if not items:
        return "Pesanan masih kosong."

    line_items = []
    total = 0

    for item_name, quantity in items.items():
        normalized_name = item_name.strip().casefold().replace("_", " ")
        if quantity < 1 or quantity > 20:
            return f"Jumlah untuk {item_name} harus antara 1 sampai 20."
        if normalized_name not in MENU:
            available_items = ", ".join(sorted(MENU))
            return f"Item {item_name!r} tidak ada di menu. Menu latihan: {available_items}."

        unit_price = MENU[normalized_name]
        item_total = unit_price * quantity
        total += item_total
        line_items.append(
            f"{quantity} x {normalized_name.title()} "
            f"({format_idr(unit_price)} per item) = {format_idr(item_total)}"
        )

    breakdown = "; ".join(line_items)
    return f"Rincian: {breakdown}. Total pesanan: {format_idr(total)}."


ORDER_SYSTEM_PROMPT = """Kamu adalah asisten kedai latihan yang menjawab dalam Bahasa Indonesia.
Kumpulkan pesanan pelanggan melalui beberapa giliran percakapan.
Jika pelanggan baru menyebutkan pesanan dan belum bilang selesai, jangan gunakan tool dulu; tanyakan: 'Ada lagi yang ingin dipesan?'
Jika pelanggan menambahkan item dan belum bilang selesai, perbarui pesanan lalu tanyakan lagi.
Jika pelanggan mengatakan sudah selesai, tidak ada lagi, cukup, atau meminta total, panggil calculate_order_total satu kali dengan semua item dan jumlah dari percakapan ini.
Jangan menebak harga, item, atau jumlah. Setelah tool mengembalikan hasil, sampaikan rincian dan total sesuai hasil tersebut.
"""

order_agent = create_agent(
    model=model,
    tools=[calculate_order_total],
    system_prompt=ORDER_SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

order_config = {"configurable": {"thread_id": "cafe-order-demo-001"}}
print("Agent pesanan siap; thread percakapan:", order_config["configurable"]["thread_id"])

### Giliran 1 — Agent bertanya apakah ada tambahan

Jalankan satu giliran. Agent seharusnya mengingat item dan bertanya apakah pelanggan ingin menambahkan sesuatu. Jawaban model bisa sedikit berbeda, tetapi belum ada total karena pelanggan belum menyatakan pesanan selesai.

In [ ]:
turn_1 = order_agent.invoke(
    {"messages": [{"role": "user", "content": "Saya pesan 2 latte."}]},
    config=order_config,
)

print(turn_1["messages"][-1].content)

### Giliran 2 — Pelanggan menambah item dan menyelesaikan pesanan

Gunakan `thread_id` yang sama. Agent sekarang punya konteks pesanan sebelumnya; pelanggan menambahkan dua item lagi dan menyatakan bahwa pesanannya sudah lengkap.

In [ ]:
turn_2 = order_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tambahkan 1 iced tea dan 2 brownies. Itu saja, tolong hitung totalnya.",
            }
        ]
    },
    config=order_config,
)

print(turn_2["messages"][-1].content)

**Hasil tool yang diharapkan:** 2 latte = Rp64.000, 1 iced tea = Rp18.000, dan 2 brownies = Rp44.000; total = **Rp126.000**. Teks jawaban agent dapat berbeda, tetapi rincian dan totalnya harus sesuai hasil tool.

### Periksa riwayat dan konteks

Cari pesan tool call dan hasil `ToolMessage`. Dua input pengguna dan pertanyaan agent sebelumnya tetap ada dalam riwayat thread yang sama. Checkpointer ini hanya menyimpan state di memori selama proses notebook aktif; restart kernel akan menghapusnya.

In [ ]:
for message in turn_2["messages"]:
    print(f"\n{type(message).__name__}")

    tool_calls = getattr(message, "tool_calls", [])
    if tool_calls:
        print("Permintaan tool:", tool_calls)

    if type(message).__name__ == "ToolMessage":
        print("Hasil tool:", message.content)

## 7. Tantangan mandiri (10–15 menit)

Ubah notebook dan catat apa yang Anda amati:

1. Tambahkan `"cappuccino": 35_000` ke `MENU`.
2. Panggil tool langsung untuk menghitung 2 cappuccino. Hasil deterministik yang diharapkan adalah `Rp70.000`.
3. Minta agent menghitung harga 2 cappuccino. Pastikan riwayat menunjukkan pemanggilan tool dan hasil `Rp70.000`.
4. Coba item yang tidak ada di menu. Apakah tool menolak input dan agent menyampaikan penolakan itu dengan jelas?
5. Jelaskan bagian mana yang harus diubah jika menu nantinya dibaca dari database.

**Starter point:** ubah dictionary `MENU` di atas dan jalankan ulang cell yang mendefinisikan tool. Setelah itu jalankan lagi cell pemanggilan tool dan agent.

> **Checkpoint 6:** Pemanggilan langsung tool memberi angka yang benar, dan agent menggunakan hasil tool untuk merespons.

## 8. Debugging singkat

| Gejala | Kemungkinan penyebab | Langkah pertama |
|---|---|---|
| `ModuleNotFoundError` | Kernel bukan dari virtual environment atau paket belum terpasang. | Pilih kernel `.venv`, lalu jalankan `python -m pip install -r requirements-notebook.txt`. |
| Error API key | `OPENAI_API_KEY` belum diatur atau tidak valid. | Jalankan ulang cell key dan buat key API yang valid. Jangan mencetak atau mengunggahnya. |
| Model tidak memanggil tool | Permintaan kurang spesifik atau model/provider tidak mendukung tool calling. | Minta secara eksplisit menggunakan tool; pastikan provider dan model mendukung tool calling. |
| Agent menyebut item tidak ada | Menu belum diperbarui atau input berbeda dari nama item. | Periksa `MENU` dan panggil tool langsung untuk melihat respons deterministiknya. |
| `429` atau batas pemakaian | Kuota atau rate limit akun API tercapai. | Periksa halaman penggunaan akun, tunggu sesuai batas, atau minta instruktur menjalankan demo. |

Jika sesi notebook lama masih menyimpan definisi lama, jalankan ulang cell dari atas setelah perubahan.

## 9. Ringkasan, self-check, dan catatan produksi

### Rangkuman

- Model menghasilkan pesan berdasarkan input.
- Prompt template membuat instruksi dan variabel input lebih terstruktur.
- Chain menyambungkan langkah tetap dengan `|`.
- `RunnableParallel` menjalankan cabang independen dengan input yang sama; `assign` menambah field ke hasilnya.
- Tool adalah fungsi dengan nama, deskripsi, dan argumen yang dapat dipahami agent.
- Agent memberi model pilihan memakai tool; runtime yang menjalankan fungsi dan meneruskan hasilnya.

### Self-check

1. Apa hasil utama dari `prompt | model | StrOutputParser()`?
2. Apakah decorator `@tool` sendiri yang memutuskan kapan fungsi dipanggil?
3. Bagian mana yang menjamin kalkulasi total harga latihan benar?
4. Apa risiko jika tool agent diberi akses untuk melakukan pembayaran tanpa persetujuan pengguna?

**Kunci jawaban singkat:**

1. Sebuah alur tetap dari input, ke pesan prompt, ke model, lalu ke string jawaban.
2. Tidak. Decorator mendaftarkan metadata/skema; model dapat meminta tool, lalu runtime agent menjalankan tool yang diizinkan.
3. Fungsi Python dan validasi input; jawaban model tetap perlu diperiksa terhadap hasil tool.
4. Tindakan salah atau tidak berizin dapat berdampak pada pengguna. Batasi izin, validasi argumen, minta persetujuan sebelum tindakan yang berpengaruh, dan simpan audit log.

### Catatan produksi

Contoh ini adalah prototipe pendidikan: menu dan FAQ RAG ditanam di kode, retriever RAG hanya mencocokkan kata, konteks pesanan multi-turn hanya tersimpan di memori selama kernel aktif, dan setiap agent hanya diberi tool yang dibutuhkan untuk langkahnya. Belum ada pengujian otomatis, autentikasi pengguna, logging, evaluasi, atau pemantauan biaya. Di sistem nyata, simpan konfigurasi dan secret dengan aman, validasi input di sisi aplikasi, batasi akses setiap tool, tangani kegagalan provider, dan buat evaluasi untuk penggunaan tool yang benar maupun salah. Untuk pembayaran atau perubahan data, jangan biarkan keputusan model menjadi satu-satunya kontrol.

### Referensi resmi

- [LangChain Python quickstart](https://docs.langchain.com/oss/python/langchain/quickstart)
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain OpenAI chat integration](https://docs.langchain.com/oss/python/integrations/chat/openai)
- [Chat prompt templates](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate)
- [OpenAI GPT-4o mini model](https://developers.openai.com/api/docs/models/gpt-4o-mini)